# 8.7 · Stacking 与 Blending / Stacking & Blending

> **课程定位 / Where this fits**
> 第 7 课，**Part 8 · 集成学习**（Part 8 收官）。
> Lesson 7, **Part 8 · Ensemble Learning** (finale).
>
> 8.6 的 voting 用**固定规则**（平均/多数）组合模型。**Stacking（堆叠）** 更进一步：训一个**元模型(meta-model)** 去**学习怎么最优地组合**各基模型的预测——相当于让模型自己决定"该信谁、信多少"。它是 Kaggle 夺冠方案的**标配**。核心难点是**防泄漏**：必须用 **OOF（out-of-fold，折外）预测**喂元模型，否则严重过拟合。
> Voting (8.6) combines models with a **fixed rule** (average/majority). **Stacking** goes further: train a **meta-model** to **learn the optimal combination** of the base models' predictions — letting a model decide "who to trust and how much". It's a staple of winning Kaggle solutions. The key difficulty is **leakage prevention**: you must feed the meta-model **OOF (out-of-fold) predictions**, or it badly overfits.
>
> 💼 **实战/面试视角**："stacking 怎么工作 / OOF 预测为什么必须 / stacking vs blending vs voting" 是集成最高阶考点。
> 💼 **Practical/interview angle:** "how stacking works / why OOF is mandatory / stacking vs blending vs voting" — the top ensemble questions.

> 📐 **符号约定 / Notation**
> - 基模型 base models —— 第一层的多个模型 / level-0 models
> - 元模型 meta-model —— 第二层组合基模型预测的模型 / level-1 combiner
> - OOF —— out-of-fold, 折外预测（每个样本由"没训练过它的模型"预测）/ out-of-fold predictions

> 💡 **面试相关 / Interview-relevant**
> - "stacking 的两层结构 / 元模型学什么"（出镜率 ★★★★★）
> - "为什么必须用 OOF 预测（防泄漏）"（出镜率 ★★★★★）
> - "stacking vs blending 的区别"（★★★★）
> - "stacking vs voting"（★★★★）
> - "元模型该用简单还是复杂模型"（★★★，通常简单）

---

## 学习目标 / Learning Objectives

1. 理解 stacking 的两层结构与元模型的角色。
   Understand stacking's two-level structure and the meta-model's role.
2. **从零**用 OOF 预测实现 stacking，看清防泄漏。
   Implement stacking from scratch with OOF predictions; see leakage prevention.
3. 演示"用训练集预测喂元模型"的泄漏后果。
   Demonstrate the leakage from feeding in-sample predictions to the meta-model.
4. 对照 sklearn `StackingRegressor`。
   Compare with sklearn's `StackingRegressor`.
5. 区分 stacking vs blending vs voting。
   Distinguish stacking vs blending vs voting.

## 目录 / TOC
1. [先建直觉：让模型学怎么组合 ⭐](#1)
2. [🏠 数据 + 基模型](#2)
3. [OOF 预测：防泄漏的核心 ⭐](#3)
4. [从零 stacking + 泄漏对照 ⭐](#4)
5. [sklearn + blending + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：让模型学怎么组合 ⭐ / Intuition: Learn the Combination

voting 假设"所有模型同等可信"（或人工设权重）。但现实里**不同模型在不同区域强**——也许 SVM 擅长某类样本、树擅长另一类。stacking 的想法：**再训一个模型（元模型），让它看着各基模型的预测、去学习"在什么情况下该信谁"**。
Voting assumes "all models equally trustworthy" (or hand-set weights). But in reality **different models excel in different regions** — maybe SVM is good on some samples, trees on others. Stacking's idea: **train another model (the meta-model) that looks at the base models' predictions and learns "when to trust whom"**.

**两层结构**：
**Two-level structure:**
- **第 0 层（基模型）**：一堆多样的模型（线性、核、树…），各自对原始特征做预测。
  **Level 0 (base models):** several diverse models (linear, kernel, trees…) each predict from the raw features.
- **第 1 层（元模型）**：以"各基模型的预测"为输入，学习最优组合，输出最终预测。常用**简单模型**（线性回归/逻辑回归）当元模型——它只需学几个组合权重，太复杂反而易过拟合。
  **Level 1 (meta-model):** takes "the base models' predictions" as input, learns the optimal combination, outputs the final prediction. Usually a **simple model** (linear/logistic regression) — it only needs to learn a few combination weights; a complex one overfits.

> stacking ≈ "可学习权重的、非线性的 voting"。
> Stacking ≈ "voting with learnable, possibly nonlinear weights".


<a id="2"></a>
## 2. 数据 + 基模型 / Data & Base Models

用 **California Housing**（回归，作为 Kaggle "House Prices" 的替身——后者需下载）。准备三个多样的基回归模型：岭回归（线性）、KNN（局部）、随机森林（树）。
Using **California Housing** (regression, standing in for Kaggle "House Prices" which needs downloading). We prepare three diverse base regressors: Ridge (linear), KNN (local), Random Forest (tree).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
sns.set_theme(style="whitegrid")

data = fetch_california_housing()
# 取子样本加速 / subsample for speed
rng = np.random.default_rng(0); idx = rng.choice(len(data.data), 6000, replace=False)
X, y = data.data[idx], data.target[idx]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

base_models = {
    "Ridge(线性)":   make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "KNN(局部)":     make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
    "RF(树)":        RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
}
print("各基模型单独的 test R²:")
for name, m in base_models.items():
    m.fit(X_tr, y_tr)
    print(f"  {name:<14} {r2_score(y_te, m.predict(X_te)):.4f}")


<a id="3"></a>
## 3. OOF 预测：防泄漏的核心 ⭐ / OOF Predictions: The Crux

stacking 最关键、也最容易错的一步：**元模型该用什么数据训练？**
The most critical and error-prone step: **what data should the meta-model train on?**

**错误做法**：让基模型在全部训练集上训练，再用它们对**同一批训练集**的预测喂元模型。问题：基模型在训练数据上"见过答案"，预测过度乐观、几乎完美——元模型学到的是"基模型在见过的数据上有多准"，这和测试时（基模型没见过的数据）完全不符 → **严重泄漏、过拟合**。
**Wrong:** train base models on all training data, then feed their predictions on the **same training data** to the meta-model. Problem: base models have "seen the answers", so their in-sample predictions are over-optimistic, nearly perfect — the meta-model learns "how good they are on data they've seen", which doesn't match test time → **severe leakage, overfitting**.

**正确做法：OOF（out-of-fold，折外）预测**。把训练集切成 K 折，对每一折，**用其它 K−1 折训练基模型、预测这一折**。这样每个训练样本的"基模型预测"都来自**没见过它的模型**——和测试时的情形一致。把这些 OOF 预测拼起来，才是给元模型的干净训练数据。这正是 7.4 嵌套 CV 思想在 stacking 里的应用。
**Right: OOF (out-of-fold) predictions.** Split the training set into K folds; for each fold, **train base models on the other K−1 folds and predict this fold**. Then each training sample's "base prediction" comes from a model that **never saw it** — matching test time. Concatenating these OOF predictions gives clean training data for the meta-model. This is the nested-CV idea (7.4) applied to stacking.


In [ ]:
def get_oof(model, X, y, X_test, n_folds=5):
    # 返回: 训练集的 OOF 预测 + 测试集的平均预测 / OOF train preds + averaged test preds
    oof = np.zeros(len(X))
    test_preds = np.zeros((n_folds, len(X_test)))
    kf = KFold(n_folds, shuffle=True, random_state=0)
    for i, (tr_idx, val_idx) in enumerate(kf.split(X)):
        m = clone_fit(model, X[tr_idx], y[tr_idx])      # 在 K-1 折上训
        oof[val_idx] = m.predict(X[val_idx])            # 预测留出的那一折(它没参与训练!)
        test_preds[i] = m.predict(X_test)               # 顺便预测测试集
    return oof, test_preds.mean(0)                       # 测试集取各折平均

from sklearn.base import clone
def clone_fit(model, X, y): return clone(model).fit(X, y)

# 为每个基模型生成 OOF(训练) 和 test 预测 / OOF features for the meta-model
oof_train = np.column_stack([get_oof(m, X_tr, y_tr, X_te)[0] for m in base_models.values()])
oof_test  = np.column_stack([get_oof(m, X_tr, y_tr, X_te)[1] for m in base_models.values()])
print(f"元模型的训练特征(OOF): {oof_train.shape}  (每列=一个基模型的折外预测)")
print("每个训练样本的基模型预测都来自'没训练过它的模型' → 无泄漏, 和测试时一致")


<a id="4"></a>
## 4. 从零 stacking + 泄漏对照 ⭐ / From-scratch Stacking & the Leakage Contrast

用 OOF 预测训元模型（线性回归），得到正确的 stacking。然后做**反面对照**：故意用"基模型对训练集的 in-sample 预测"训元模型——看它的 CV/train 估计如何被泄漏吹得虚高、而 test 反而更差。
Train the meta-model (linear regression) on OOF predictions for correct stacking. Then a **counter-example**: deliberately train the meta-model on the base models' in-sample training predictions — and watch its train estimate inflated by leakage while test actually suffers.


In [ ]:
# ✅ 正确 stacking: 元模型在 OOF 预测上训 / meta-model on OOF predictions
meta = LinearRegression().fit(oof_train, y_tr)
stack_pred = meta.predict(oof_test)
print(f"✅ 正确 stacking (OOF): test R² = {r2_score(y_te, stack_pred):.4f}")
print(f"   元模型学到的组合权重: {dict(zip(base_models.keys(), meta.coef_.round(3)))}")
print("   (权重反映元模型'信谁多一点'; 这里 RF 通常权重最大)\n")

# ❌ 错误: 基模型全量训练后, 用 in-sample 预测喂元模型 / WRONG: in-sample predictions leak
insample_train = np.column_stack([m.fit(X_tr, y_tr).predict(X_tr) for m in base_models.values()])
insample_test  = np.column_stack([m.predict(X_te) for m in base_models.values()])
meta_leak = LinearRegression().fit(insample_train, y_tr)
print(f"❌ 泄漏 stacking (in-sample): 元模型在训练特征上 R² = {meta_leak.score(insample_train, y_tr):.4f} ← 虚高!")
print(f"   它的真实 test R² = {r2_score(y_te, meta_leak.predict(insample_test)):.4f}")
print("\n泄漏版: 训练估计被吹得虚高(基模型 in-sample 近乎完美), 元模型被骗 → 真实 test 不如正确版")
print("→ 这就是为什么 stacking 必须用 OOF 预测(7.4 嵌套 CV 思想)")


<a id="5"></a>
## 5. sklearn + blending + 小结 ⭐ / sklearn, Blending & Summary

sklearn 的 `StackingRegressor` 内部自动用交叉折生成 OOF 预测，一行搞定。
sklearn's `StackingRegressor` automatically generates OOF predictions via cross-folds — one line.

**Blending** 是 stacking 的简化版（面试常对比）：不做 K 折 OOF，而是**单独留出一个 holdout 集**，基模型在剩余数据上训、在 holdout 上预测，用这些 holdout 预测训元模型。**更简单更快、不会折间泄漏**，但**浪费了一块数据**（holdout 不参与基模型最终训练）、且元模型见的数据更少。
**Blending** is a simpler variant (often contrasted): instead of K-fold OOF, **hold out a single set**; base models train on the rest and predict the holdout, and the meta-model trains on those holdout predictions. **Simpler, faster, no fold leakage**, but **wastes a chunk of data** and the meta-model sees less.


In [ ]:
from sklearn.ensemble import StackingRegressor

# sklearn StackingRegressor: 内部自动 OOF / handles OOF internally
stack = StackingRegressor(
    estimators=list(base_models.items()),
    final_estimator=LinearRegression(), cv=5)
stack.fit(X_tr, y_tr)
print(f"sklearn StackingRegressor test R² = {r2_score(y_te, stack.predict(X_te)):.4f}")

# 对比所有方案 / compare everything
print(f"\n{'方法':<26}{'test R²':>9}")
for name, m in base_models.items():
    print(f"{name+' (单模型)':<28}{r2_score(y_te, m.fit(X_tr,y_tr).predict(X_te)):>9.4f}")
from sklearn.ensemble import VotingRegressor
vote = VotingRegressor(list(base_models.items())).fit(X_tr, y_tr)
print(f"{'平均 Voting':<28}{r2_score(y_te, vote.predict(X_te)):>9.4f}")
print(f"{'Stacking (正确 OOF)':<28}{r2_score(y_te, stack_pred):>9.4f}")
print("\nStacking 通常 ≥ Voting ≥ 最好单模型(若基模型多样); 但提升常是小数点后几位")
print("代价: stacking 复杂度/训练成本更高, 可解释性更差 → 竞赛常用, 生产看收益是否值得")


```
Stacking 两层: 第0层多个多样基模型 → 第1层元模型学习最优组合各基模型的预测
元模型通常用简单模型(线性/逻辑回归), 只学组合权重, 复杂易过拟合
OOF 防泄漏 ⭐: 元模型必须用折外预测训(每样本由没训过它的模型预测), 否则严重泄漏
  错误用 in-sample 预测 → 训练估计虚高、真实 test 反而更差
Blending: 用单独 holdout 代替 K 折 OOF; 更简单更快, 但浪费一块数据
效果: Stacking ≥ Voting ≥ 最好单模型(若基多样); 提升常很小, 代价是复杂度↑
voting(固定规则) < stacking(学习组合); 竞赛常用 stacking, 生产权衡收益
```

### 💡 面试速查 / Interview cheat-sheet
1. **Stacking = 两层**: 基模型 + 元模型(学习怎么组合); 元模型用简单模型。
   Stacking = two levels: base models + a meta-model (learns the combination); keep the meta-model simple.
2. **必须用 OOF 预测**训元模型(防泄漏); in-sample 预测会严重过拟合。
   Train the meta-model on OOF predictions (no leakage); in-sample predictions badly overfit.
3. **Stacking(K折OOF) vs Blending(单holdout)**: 后者更简单更快但浪费数据。
   Stacking (K-fold OOF) vs blending (single holdout): the latter is simpler/faster but wastes data.
4. **Stacking vs Voting**: voting 固定规则, stacking 学习组合(更强但更复杂)。
   Voting uses a fixed rule; stacking learns the combination (stronger but more complex).
5. 提升常很小, 复杂度高 → 竞赛常用, 生产权衡是否值得。
   Gains are often small with high complexity → common in competitions, weigh it for production.

### Part 8 完成 🎉
集成学习全部走通: bagging(降方差) → RF 深入 → AdaBoost → GBDT 推导 → 三大 boosting 库 → voting → stacking。两大范式(bagging/boosting)的原理、从零实现、和现代库的工程优化全部打通——这是表格数据建模的核心竞争力。
Part 8 complete: bagging → RF deep dive → AdaBoost → GBDT derivation → the boosting trio → voting → stacking. The two paradigms (bagging/boosting), from-scratch derivations, and modern libraries' engineering — the core competency for tabular modeling.
